# aether — Colab preflight

Первый этап: английское голосовое демо, Google Drive, бюджет 500 юнитов.
Этот notebook готовит среду и отчёт ресурсов. Модель и обучение ещё недоступны.
Откройте в **удалённом управляемом runtime платного Colab**, не в local runtime.
Выделение GPU расходует бюджет даже без обучения: запускайте только вручную.
Для первого BF16 baseline ориентир — GPU с 24+ ГБ, но это не гарантия размещения.
Сначала разрешён только preflight; никаких загрузок весов или optimizer здесь нет.


In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

CONFIRM_REMOTE_PAID_COLAB = False  # Подтвердите среду в интерфейсе Colab вручную.
RUN_TRAINING = False  # Оставить False; True тоже не разрешает обучение.
BUDGET_UNITS = 500
RUN_ID = ""  # Например english-demo-preflight-001; уникальное имя.
DRIVE_ROOT = "/content/drive/MyDrive/aether"
SOURCE_SHA256 = ""  # Скопируйте из dist/aether-source.sha256.
UV_VERSION = "0.12.13"

## Проверка выбора среды
Подтверждение ниже — заявление пользователя, не доказательство инфраструктуры.
Наблюдения среды будут записаны отдельно. Обучающий gate остаётся закрытым.


In [ ]:
if RUN_TRAINING:
    raise RuntimeError("Training is unavailable; keep RUN_TRAINING=False")
if not CONFIRM_REMOTE_PAID_COLAB:
    raise RuntimeError("Select a managed remote paid Colab runtime and confirm above")
__import__("google.colab")
if not Path("/content").is_dir() or sys.platform != "linux":
    raise RuntimeError("Expected Colab Linux environment; local runtime is forbidden")
if not RUN_ID or len(SOURCE_SHA256) != 64:
    raise ValueError("Set unique RUN_ID and exact source archive SHA-256")

## Получение точной версии кода
Загрузите `dist/aether-source.zip` из подготовленного проекта. Архив включает
текущие незакоммиченные изменения; его SHA-256 фиксирует точную версию.
Распаковка повторного запуска использует тот же неизменный каталог исходников.


In [ ]:
from io import BytesIO
from zipfile import ZipFile

from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload only aether-source.zip")
source_bytes = next(iter(uploaded.values()))
if hashlib.sha256(source_bytes).hexdigest() != SOURCE_SHA256:
    raise ValueError("Source archive SHA-256 mismatch")
PROJECT = Path("/content") / ("aether-" + SOURCE_SHA256[:16])
with ZipFile(BytesIO(source_bytes)) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)) or sum(i.file_size for i in archive.infolist()) > 20_000_000:
        raise ValueError("Invalid source archive size or duplicate paths")
    manifest = json.loads(archive.read("source-manifest.json"))
    if set(names) != set(manifest["files"]) | {"source-manifest.json"}:
        raise ValueError("Archive manifest mismatch")
    for name, expected_hash in manifest["files"].items():
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Unsafe source path")
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != expected_hash:
            raise ValueError("Source file hash mismatch")
        destination = PROJECT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists():
            if destination.read_bytes() != data:
                raise ValueError("Existing source snapshot changed")
        else:
            destination.write_bytes(data)
print("Verified source snapshot:", SOURCE_SHA256)

## uv и отдельное окружение
В системное ядро устанавливается только зафиксированный uv. Зависимости пакета
управляются `uv sync --locked`; Python пакета — 3.12.14. Модельные зависимости
пока не выбираются и не скачиваются.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv==" + UV_VERSION], check=True)
UV = [sys.executable, "-m", "uv"]
subprocess.run(
    UV + ["sync", "--locked", "--no-dev", "--python", "3.12.14"], cwd=PROJECT, check=True
)


def package_run(*arguments):
    return subprocess.run(
        UV + ["run", "--locked", "--no-dev", *arguments],
        cwd=PROJECT,
        check=True,
        capture_output=True,
        text=True,
    )


print(package_run("aether", "--version").stdout)
print(
    package_run("aether", "train", "--config", "configs/model/tiny.json", "--validate-only").stdout
)

## Preflight ресурсов
Процесс пакета читает CPU/RAM/диск и `nvidia-smi`. Не загружает модель, не делает
forward и не проверяет тариф программно. Расход юнитов смотрите в интерфейсе Colab.


In [ ]:
result = package_run(
    "python",
    "-c",
    "import json; from aether.preflight import collect_preflight; "
    "print(json.dumps(collect_preflight()))",
)
report = json.loads(result.stdout)
report.update(
    source_sha256=SOURCE_SHA256,
    uv_version=UV_VERSION,
    budget_units=BUDGET_UNITS,
    language="en",
    scenario="chat_demo",
    user_asserted_remote_paid_colab=CONFIRM_REMOTE_PAID_COLAB,
)
print(json.dumps(report, indent=2, ensure_ascii=False))

## Google Drive и сохранение отчёта
Авторизуйте Drive. Создаётся новый `runs/RUN_ID`; существующий запуск не
перезаписывается. Для повторного preflight задайте новое RUN_ID.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Drive is not mounted")
LOCAL_REPORT = PROJECT / "preflight.json"
LOCAL_REPORT.write_text(json.dumps(report, indent=2), encoding="utf-8")
result = package_run(
    "python",
    "-c",
    "from pathlib import Path; import sys; from aether.storage import create_run; "
    "run=create_run(Path(sys.argv[1]),sys.argv[2]); "
    "(run/'preflight.json').write_bytes(Path(sys.argv[3]).read_bytes()); print(run)",
    DRIVE_ROOT,
    RUN_ID,
    str(LOCAL_REPORT),
)
print("Report saved:", result.stdout)
files.download(str(LOCAL_REPORT))

## Артефакты
Выбран кандидат BF16 с начальным кодеком и текстовым токенизатором. Реальная загрузка требует фиксации revision, хешей, архитектуры и проверки совместимости. В этой версии загрузка весов отсутствует.


## Baseline
Пока заблокирован A07/A10–A13/A19. Пришлите preflight.json; модельные операции здесь не подменяются синтетическим демо.


## Pilot / resume / train
Обучение закрыто, RUN_TRAINING=False. Хранилище checkpoint проверено на локальных фикстурах, но восстановление тренера и Colab Drive ещё не проверены. Нужны модель, данные, baseline и все зависимости A22.


## Evaluate / export
Будут доступны после реализации модели и оценки. Этот notebook сохраняет только preflight.json, а не модель или checkpoint.


In [ ]:
if RUN_TRAINING:
    raise RuntimeError("Training is blocked pending verified runtime, model, data and baseline")
print("Preflight complete. Send preflight.json for review; then disconnect the GPU runtime.")